# Encoding Categorical Variables

**Topic:** Data Preprocessing

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox, HTML
from IPython.display import display, clear_output

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** why machine learning models cannot work with raw text categories like `"male"`, `"female"`, or `"Southampton"`
- **Explain** the difference between label encoding, one-hot encoding, and ordinal encoding and when each is appropriate
- **Interpret** how the wrong encoding method can silently introduce a false ordering that misleads a model

> **Tip:** Select `"sex"` from the column dropdown and apply Label Encoding first. Read the warning carefully. Then switch to One-Hot Encoding and notice how the column count changes.

---
## How we got here

In `02_handling_missing_data.ipynb` we handled the missing values in `age` and `cabin`. Those columns are now numeric and complete. But three columns remain unusable: `sex`, `embarked`, and `pclass` all contain text or category values that no algorithm can do math on.

This notebook turns those text categories into numbers, and shows why the method you choose matters.

---
## Why this matters for data science

Most real-world datasets contain categorical columns: customer segments, city names, product types, blood types, education levels. Choosing the wrong encoding method can silently introduce a false ordering into your data, making a model believe "Southampton" is mathematically greater than "Cherbourg." The model has no way to know that is meaningless; it just follows the numbers.

---
## Try it yourself

In [2]:
out = Output()

COLUMNS = {
    "sex": {
        "dtype": "nominal",
        "categories": sorted(titanic["sex"].dropna().unique().tolist()),
        "why_ordinal_wrong": "There is no meaningful numeric order between male and female.",
    },
    "embarked": {
        "dtype": "nominal",
        "categories": sorted(titanic["embarked"].dropna().unique().tolist()),
        "why_ordinal_wrong": "Ports of embarkation (C, Q, S) have no inherent numeric ranking.",
    },
    "pclass": {
        "dtype": "ordinal",
        "categories": sorted(titanic["pclass"].dropna().unique().tolist()),
        "why_ordinal_wrong": "",
    },
}

col_dd = Dropdown(
    options=list(COLUMNS.keys()),
    value="sex",
    description="Column:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="260px"),
)
enc_dd = Dropdown(
    options=["Label Encoding", "One-Hot Encoding", "Ordinal Encoding"],
    value="Label Encoding",
    description="Encoding:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px"),
)

def render(change=None):
    col      = col_dd.value
    encoding = enc_dd.value
    info     = COLUMNS[col]
    series   = titanic[col].dropna().head(20).reset_index(drop=True).astype(str)

    warning_html = ""
    if encoding == "Label Encoding" and info["dtype"] == "nominal":
        warning_html = (
            '<div style="background:#fff3cd;border-left:4px solid #ffc107;'
            'padding:10px 14px;border-radius:4px;margin-top:8px;font-size:13px;">'
            '⚠️ <strong>Warning: nominal variable detected:</strong> '
            '"' + col + '" has no natural numeric order. Label encoding assigns '
            '0, 1, 2… implying an ordering that does not exist. '
            + info["why_ordinal_wrong"] + ' '
            'Use One-Hot Encoding for nominal variables.</div>'
        )

    if encoding == "Label Encoding":
        le = LabelEncoder()
        encoded     = le.fit_transform(series)
        header_vals = [col, "label_encoded"]
        cell_vals   = [series.tolist(), encoded.tolist()]
        title       = f"Label Encoding: '{col}'"
        note        = "Classes: " + str(dict(enumerate(le.classes_)))

    elif encoding == "One-Hot Encoding":
        ohe = OneHotEncoder(sparse_output=False, dtype=int)
        encoded     = ohe.fit_transform(series.values.reshape(-1, 1))
        ohe_cols    = [f"{col}_{c}" for c in ohe.categories_[0]]
        header_vals = [col] + ohe_cols
        cell_vals   = [series.tolist()] + [encoded[:, i].tolist() for i in range(encoded.shape[1])]
        title       = f"One-Hot Encoding: '{col}' → {len(ohe_cols)} binary columns"
        note        = f"Original 1 column → {len(ohe_cols)} binary columns. No false ordering."

    else:  # Ordinal
        cats = [[str(c) for c in info["categories"]]]
        oe = OrdinalEncoder(categories=cats)
        encoded     = oe.fit_transform(series.values.reshape(-1, 1)).flatten().astype(int)
        header_vals = [col, "ordinal_encoded"]
        cell_vals   = [series.tolist(), encoded.tolist()]
        title       = f"Ordinal Encoding: '{col}'"
        note        = "Explicit order: " + str(info["categories"])

    n_rows     = len(series)
    row_colors = ["#ffffff" if r % 2 == 0 else PALETTE["background"] for r in range(n_rows)]
    cell_colors = [row_colors for _ in header_vals]

    fig = go.Figure(go.Table(
        header=dict(
            values=[f"<b>{h}</b>" for h in header_vals],
            fill_color=PALETTE["primary"],
            font=dict(color="white", size=12),
            align="left",
        ),
        cells=dict(
            values=cell_vals,
            fill_color=cell_colors,
            align="left",
            font=dict(size=12),
        ),
    ))
    layout = base_layout(title=title)
    layout.update(height=500, margin=dict(l=20, r=20, t=60, b=20))

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(data=fig.data, layout=layout))
        if warning_html:
            display(HTML(warning_html))
        display(HTML(f'<p style="font-size:13px;color:#555;margin-top:6px;">{note}</p>'))

col_dd.observe(lambda c: render(), names="value")
enc_dd.observe(lambda c: render(), names="value")
display(VBox([HBox([col_dd, enc_dd]), out]))
render()

---
## What's happening?

Machine learning models are mathematical functions. They can only compute with numbers. When a column contains text like `"male"` or `"Southampton"`, the algorithm has no idea what to do with it.

**Label encoding** converts each unique category to an integer (0, 1, 2, ...). This is fast and compact, but it implies an ordering: if `C=0`, `Q=1`, `S=2`, the model may think Southampton is twice as large as Cherbourg. For nominal variables that have no natural order, this is misleading.

**One-hot encoding** creates a separate binary column for each category. There is no implied ordering: each column simply says "is this passenger from Southampton: yes or no." The downside is that it creates more columns, which can become expensive for columns with many unique values.

**Watch out: the dummy variable trap.** When you one-hot encode a column with N categories, you get N binary columns. But they are redundant: if a passenger is not from Southampton and not from Queenstown, they must be from Cherbourg. Knowing any N-1 columns tells you the Nth. This perfect redundancy (called multicollinearity) causes algorithms that assume a linear relationship to struggle to find unique coefficients. The fix is to drop one column: `OneHotEncoder(drop='first')`. The remaining N-1 columns still represent every category, and the redundancy is gone. Tree-based algorithms are unaffected by multicollinearity, so the drop is not needed for them.

**Ordinal encoding** works like label encoding but lets you specify the correct order explicitly. Use it when the categories genuinely have a ranking, like `pclass` (1st > 2nd > 3rd) or education level.

> **Discussion question:** Why would label encoding `"sex"` as 0/1 be misleading to a model compared to one-hot encoding?

| Encoding type | When to use it | Risk if misused | sklearn class |
|---|---|---|---|
| Label Encoding | Binary categories only | Implies false ordering for nominal variables | `LabelEncoder` |
| One-Hot Encoding | Nominal categories with no natural order | Creates many columns for high-cardinality features; causes multicollinearity in linear models unless one column is dropped | `OneHotEncoder` |
| Ordinal Encoding | Categories with a meaningful rank order | Wrong order assignment gives the model bad information | `OrdinalEncoder` |

---
## When you need this step

All algorithms require numeric input. Without encoding, any column containing strings will cause a `ValueError` before training begins. This step is non-negotiable.

## When the encoding *method* is model-dependent

The fact that you need encoding is universal, but which encoding method you choose depends on your algorithm.

- **Algorithms that assume features have mathematical meaning** (such as those that find a best-fit line through the data) treat the difference between 0 and 1, and between 1 and 2, as meaningful distances. For these, one-hot encoding is safer: it never implies a numeric ordering that does not exist.
- **Algorithms that split data on thresholds** (asking "is this value above or below X?") do not care whether `"male"` is encoded as 0 or 1 or whether it is split into separate binary columns. Either encoding method works: the tree finds its own boundary.

The practical rule: **default to one-hot encoding** for nominal columns. You will rarely go wrong. As you meet specific algorithms, you will learn when ordinal encoding is preferred.

---
## Key takeaway

> **Label encoding implies an order that may not exist. For nominal categories like city names or passenger sex, one-hot encoding is almost always the safer choice.**

---
*Next up: 04_feature_scaling.ipynb, now that categories are numbers, we need all numbers to live on the same scale*